In [ ]:
# this script extracts text from PDFs, cleans it by removing references and page numbers, and saves the cleaned text to .txt files for use in a RAG pipeline.

import fitz   # PyMuPDF
import os
import re

INPUT_DIR  = "Selected papers"
OUTPUT_DIR = "Cleaned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Phrases that signal the start of the references section
REF_PATTERNS = [
    r'\nReferences\n',
    r'\nBIBLIOGRAPHY\n',
    r'\nREFERENCES\n',
    r'\nReference\n',
]

def extract_text(pdf_path):
    doc  = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def clean_text(text):
    # Cut off references section — everything after is noise
    for pattern in REF_PATTERNS:
        match = re.search(pattern, text)
        if match:
            text = text[:match.start()]
            break

    # Remove standalone page numbers (e.g. just "12" on a line)
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)

    # Collapse more than 2 consecutive blank lines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove lines that are purely dashes or underscores (dividers)
    text = re.sub(r'\n[-_]{3,}\n', '\n', text)

    return text.strip()

files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.pdf')]
print(f"Found {len(files)} PDFs to process...\n")

for fname in sorted(files):
    pdf_path = os.path.join(INPUT_DIR, fname)
    txt_name = fname.replace('.pdf', '.txt')
    out_path = os.path.join(OUTPUT_DIR, txt_name)

    try:
        raw   = extract_text(pdf_path)
        clean = clean_text(raw)

        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(clean)

        word_count = len(clean.split())
        print(f"  OK  {fname:50s}  →  {word_count:,} words")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print("\nCleaning complete. Check the Cleaned/ folder.")

Found 15 PDFs to process...

  OK  00_nested_learning_original.pdf                     →  26,626 words
  OK  03_fast_weight_programming.pdf                      →  13,750 words
  OK  04_end_to_end_ttt_long_context.pdf                  →  12,128 words
  OK  05_dynamic_nested_hierarchies.pdf                   →  4,862 words
  OK  06_crad_hope_anomaly_detection.pdf                  →  8,253 words
  OK  08_tiny_autoregressive_recursive.pdf                →  4,750 words
  OK  10_hydra_dual_exponentiated_memory.pdf              →  6,018 words
  OK  11_beyond_ttt_optimal_control.pdf                   →  8,287 words
  OK  13_allmem_long_context.pdf                          →  4,971 words
  OK  14_vision_transformers_never_stop.pdf               →  4,941 words
  OK  15_msa_memory_sparse_attention.pdf                  →  7,680 words
  OK  16_prompt_injection_nested_learning.pdf             →  11,109 words
  OK  17_memory_caching_rnns.pdf                          →  8,389 words
  OK  22_learning_

In [ ]:
# this cell prints the statistics of the cleaned corpus, including total words, characters, sentences, and averages per document.

import os
import re

CLEANED_DIR = "Cleaned"

files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith('.txt')])

print(f"{'='*70}")
print(f"  CORPUS STATISTICS — {len(files)} documents")
print(f"{'='*70}")
print(f"  {'File':<45} {'Words':>7}  {'Chars':>8}  {'Sentences':>10}")
print(f"  {'-'*70}")

total_words     = 0
total_chars     = 0
total_sentences = 0
all_word_counts = []

for fname in files:
    fpath = os.path.join(CLEANED_DIR, fname)
    with open(fpath, 'r', encoding='utf-8') as f:
        text = f.read()

    words     = len(text.split())
    chars     = len(text)
    sentences = len(re.findall(r'[.!?]+', text))

    total_words     += words
    total_chars     += chars
    total_sentences += sentences
    all_word_counts.append(words)

    print(f"  {fname:<45} {words:>7,}  {chars:>8,}  {sentences:>10,}")

print(f"\n{'='*70}")
print(f"  TOTALS")
print(f"{'='*70}")
print(f"  Documents         : {len(files)}")
print(f"  Total words       : {total_words:,}")
print(f"  Total characters  : {total_chars:,}")
print(f"  Total sentences   : {total_sentences:,}")
print(f"  Avg words/doc     : {total_words // len(files):,}")
print(f"  Min words/doc     : {min(all_word_counts):,}  ({files[all_word_counts.index(min(all_word_counts))]})")
print(f"  Max words/doc     : {max(all_word_counts):,}  ({files[all_word_counts.index(max(all_word_counts))]})")
print(f"{'='*70}")

  CORPUS STATISTICS — 15 documents
  File                                            Words     Chars   Sentences
  ----------------------------------------------------------------------
  00_nested_learning_original.txt                26,626   168,020       2,001
  03_fast_weight_programming.txt                 13,750    86,115         813
  04_end_to_end_ttt_long_context.txt             12,128    73,379         869
  05_dynamic_nested_hierarchies.txt               4,862    33,548         416
  06_crad_hope_anomaly_detection.txt              8,253    58,586       1,223
  08_tiny_autoregressive_recursive.txt            4,750    32,244         286
  10_hydra_dual_exponentiated_memory.txt          6,018    37,740         785
  11_beyond_ttt_optimal_control.txt               8,287    53,567         687
  13_allmem_long_context.txt                      4,971    35,316         442
  14_vision_transformers_never_stop.txt           4,941    33,622         360
  15_msa_memory_sparse_attention.t

In [ ]:
# ============================================================
# RAG Corpus Builder — All 3 Chunking Strategies
# NLP with Deep Learning — Assignment 3
# Run on Kaggle with GPU OFF (CPU is fine for this step)
# ============================================================
#
# BEFORE RUNNING:
#   1. Upload your cleaned/ folder as a Kaggle dataset
#   2. In the notebook, go to:
#      Add Data → Your Datasets → select your cleaned papers dataset
#   3. Your files will appear at /kaggle/input/<dataset-name>/
#   4. Update CLEANED_DIR below to match that path
#
# OUTPUT:
#   /kaggle/working/corpus_A_fixed.pkl
#   /kaggle/working/corpus_B_recursive.pkl
#   /kaggle/working/corpus_C_semantic.pkl
#   /kaggle/working/chunking_summary.txt
# ============================================================

# ── CELL 1: Install dependencies ─────────────────────────────────────────────

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "langchain",
    "langchain-experimental",
    "langchain-huggingface",
    "sentence-transformers"
], check=True)

print("✓ Dependencies installed")

# ── CELL 2: Imports ───────────────────────────────────────────────────────────

import os
import pickle
import time
import warnings
warnings.filterwarnings("ignore")

from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

print("✓ Imports done")

# ── CELL 3: Configuration ─────────────────────────────────────────────────────

# UPDATE THIS PATH to match where your cleaned .txt files are on Kaggle
# Example: "/kaggle/input/nl-papers-cleaned/cleaned"
CLEANED_DIR = "/kaggle/input/your-dataset-name/cleaned"

OUTPUT_DIR  = "/kaggle/working"   # Kaggle saves here — do not change

# Verify cleaned dir exists and has files
txt_files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith('.txt')])
print(f"✓ Found {len(txt_files)} cleaned .txt files in {CLEANED_DIR}")
for f in txt_files:
    size = os.path.getsize(os.path.join(CLEANED_DIR, f))
    print(f"   {f:55s}  {size/1024:.1f} KB")

# ── CELL 4: Paper metadata ────────────────────────────────────────────────────

PAPER_META = {
    "00_nested_learning_original": {
        "title":    "Nested Learning: The Illusion of Deep Learning Architectures",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Behrouz, Razaviyayn, Zhong, Mirrokni",
    },
    "03_fast_weight_programming": {
        "title":    "Fast Weight Programming and Linear Transformers: From ML to Neurobiology",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Irie, Gershman",
    },
    "04_end_to_end_ttt_long_context": {
        "title":    "End-to-End Test-Time Training for Long Context",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Tandon, Dalal, Li et al.",
    },
    "05_dynamic_nested_hierarchies": {
        "title":    "Dynamic Nested Hierarchies: Self-Evolution in ML for Lifelong Intelligence",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Akbar Anbar Jafari, Ozcinar, Anbarjafari",
    },
    "06_crad_hope_anomaly_detection": {
        "title":    "CRAD-HOPE: Brain-inspired Nested Learning Framework for Few-Shot Anomaly Detection",
        "year":     2026,
        "category": "Application",
        "authors":  "Cao, Cao, Wu",
    },
    "08_tiny_autoregressive_recursive": {
        "title":    "Tiny Autoregressive Recursive Models",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Rauba, Fanconi, van der Schaar",
    },
    "10_hydra_dual_exponentiated_memory": {
        "title":    "HYDRA: Dual Exponentiated Memory for Multivariate Time Series Analysis",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Meskin, Mirrokni, Najar, Behrouz",
    },
    "11_beyond_ttt_optimal_control": {
        "title":    "Beyond Test-Time Training: Learning to Reason via Hardware-Efficient Optimal Control",
        "year":     2026,
        "category": "Core Extension",
        "authors":  "Wang, Yang, Wang, Xiao, Liu et al.",
    },
    "13_allmem_long_context": {
        "title":    "AllMem: A Memory-centric Recipe for Efficient Long-context Modeling",
        "year":     2026,
        "category": "Application",
        "authors":  "Wang, Wang, Peng, Qin et al.",
    },
    "14_vision_transformers_never_stop": {
        "title":    "Vision Transformers That Never Stop Learning",
        "year":     2026,
        "category": "Application",
        "authors":  "Sun, Yuan, Wang, Chen",
    },
    "15_msa_memory_sparse_attention": {
        "title":    "MSA: Memory Sparse Attention for Efficient End-to-End Memory Model Scaling",
        "year":     2026,
        "category": "Application",
        "authors":  "Chen, Chen, Yi, Zhao et al.",
    },
    "16_prompt_injection_nested_learning": {
        "title":    "Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability",
        "year":     2026,
        "category": "Application",
        "authors":  "Gosmar, Dah",
    },
    "17_memory_caching_rnns": {
        "title":    "Memory Caching: RNNs with Growing Memory",
        "year":     2026,
        "category": "Core Extension",
        "authors":  "Behrouz, Li, Deng, Zhong, Razaviyayn, Mirrokni",
    },
    "22_learning_to_learn_at_scale": {
        "title":    "Learning-to-Learn at Scale: Promises and Challenges",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Wang, Wang, Chen, Yu, Gan, Liu",
    },
    "26_attention_not_retention": {
        "title":    "Attention Is Not Retention: The Orthogonality Constraint in Infinite-Context Architectures",
        "year":     2025,
        "category": "Related Theory",
        "authors":  "Zahn, Beton, Chana",
    },
}

print(f"✓ Metadata loaded for {len(PAPER_META)} papers")

# ── CELL 5: Define the three splitters ───────────────────────────────────────

# --- Strategy A: Fixed Character Splitting ---
# Cuts at exactly 512 characters on newline boundaries
# No awareness of sentence or paragraph structure
# Fast but produces lowest quality chunks — used as ablation baseline
splitter_A = CharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separator="\n",
    length_function=len,
)

# --- Strategy B: Recursive Character Splitting ---
# Tries \n\n → \n → ". " → " " in order, picks nearest natural boundary
# Preserves paragraph and sentence structure wherever possible
# Best balance of quality + speed — used as production system
splitter_B = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "],
    length_function=len,
)

# --- Strategy C: Semantic Chunking ---
# Embeds every sentence using all-MiniLM-L6-v2
# Splits where cosine similarity between adjacent sentences drops sharply
# Most intelligent — cuts only when the topic genuinely changes
# Slowest — takes 10-20 min on CPU across 15 papers
print("Loading embedding model for semantic chunking (this takes ~2 min)...")
t0 = time.time()
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 64},
)
print(f"✓ Embedding model loaded in {time.time()-t0:.1f}s")

splitter_C = SemanticChunker(
    embeddings=embed_model,
    breakpoint_threshold_type="percentile",
    # Splits where similarity drops below the 70th percentile
    # Lower value = fewer, larger chunks
    # Higher value = more, smaller chunks
    breakpoint_threshold_amount=70,
)

print("✓ All three splitters ready")

# ── CELL 6: Core build function ───────────────────────────────────────────────

def build_corpus(splitter, label, description, min_chunk_len=80):
    """
    Chunks all cleaned .txt files using the given splitter,
    attaches metadata to each chunk, saves as a pkl file.

    min_chunk_len: drops any chunk shorter than this (catches
    micro-chunks from abbreviation mis-splits in semantic mode)
    """
    all_chunks   = []
    all_metadata = []
    per_paper    = {}   # for the summary report

    files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith('.txt')])

    print(f"\n{'='*65}")
    print(f"  Strategy {label}: {description}")
    print(f"{'='*65}")

    t_start = time.time()

    for fname in files:
        key       = fname.replace('.txt', '')
        meta_base = PAPER_META.get(key, {
            "title":    key,
            "year":     "unknown",
            "category": "unknown",
            "authors":  "unknown",
        })

        fpath = os.path.join(CLEANED_DIR, fname)
        with open(fpath, 'r', encoding='utf-8') as f:
            text = f.read()

        # Run the splitter
        t0     = time.time()
        chunks = splitter.split_text(text)
        t1     = time.time()

        # Drop micro-chunks (< min_chunk_len chars)
        # Mostly affects Semantic strategy due to abbreviation mis-splits
        before_filter = len(chunks)
        chunks = [c.strip() for c in chunks if len(c.strip()) >= min_chunk_len]
        dropped = before_filter - len(chunks)

        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_metadata.append({
                "source":      fname,
                "title":       meta_base["title"],
                "authors":     meta_base["authors"],
                "year":        meta_base["year"],
                "category":    meta_base["category"],
                "chunk_id":    i,
                "chunk_total": len(chunks),
                "strategy":    label,
            })

        avg_len = int(sum(len(c) for c in chunks) / len(chunks)) if chunks else 0
        per_paper[fname] = {
            "chunks": len(chunks),
            "dropped": dropped,
            "avg_len": avg_len,
            "time": round(t1 - t0, 2),
        }

        drop_note = f"  ({dropped} micro-chunks dropped)" if dropped > 0 else ""
        print(f"  {fname:52s}  {len(chunks):3d} chunks  avg={avg_len}c  {t1-t0:.1f}s{drop_note}")

    # ── Save pkl ──────────────────────────────────────────────────────────────
    output_path = os.path.join(OUTPUT_DIR, f"corpus_{label}.pkl")
    with open(output_path, 'wb') as f:
        pickle.dump({
            "chunks":      all_chunks,
            "metadata":    all_metadata,
            "strategy":    label,
            "description": description,
        }, f)

    total_chars = sum(len(c) for c in all_chunks)
    avg_overall = total_chars // len(all_chunks) if all_chunks else 0
    elapsed     = time.time() - t_start
    file_size   = os.path.getsize(output_path) / 1024

    print(f"\n  ✓ Total chunks   : {len(all_chunks)}")
    print(f"  ✓ Avg chunk size : {avg_overall} characters")
    print(f"  ✓ Total time     : {elapsed:.1f}s")
    print(f"  ✓ Saved          : corpus_{label}.pkl  ({file_size:.1f} KB)")

    return {
        "label":       label,
        "description": description,
        "total_chunks": len(all_chunks),
        "avg_chars":   avg_overall,
        "elapsed":     round(elapsed, 1),
        "file_size_kb": round(file_size, 1),
        "per_paper":   per_paper,
    }

# ── CELL 7: Run all three strategies ─────────────────────────────────────────
# Total expected runtime:
#   Strategy A : ~5  seconds
#   Strategy B : ~8  seconds
#   Strategy C : ~15 minutes  (embedding every sentence)

print("Starting chunking pipeline...\n")

results = {}

results["A"] = build_corpus(
    splitter_A,
    label="A_fixed",
    description="Fixed Character Splitting  (512 chars / 50 overlap)",
)

results["B"] = build_corpus(
    splitter_B,
    label="B_recursive",
    description="Recursive Character Splitting  (400 chars / 80 overlap)",
)

results["C"] = build_corpus(
    splitter_C,
    label="C_semantic",
    description="Semantic Chunking  (percentile=70 / all-MiniLM-L6-v2)",
)

# ── CELL 8: Print final comparison summary ────────────────────────────────────

print(f"\n{'='*65}")
print(f"  CHUNKING COMPLETE — FINAL SUMMARY")
print(f"{'='*65}")
print(f"  {'Strategy':<40} {'Chunks':>8}  {'Avg Size':>10}  {'Time':>8}  {'File':>8}")
print(f"  {'-'*65}")

for key in ["A","B","C"]:
    r = results[key]
    print(
        f"  {r['label'] + ' — ' + r['description'][:28]:<40}"
        f"  {r['total_chunks']:>6}"
        f"  {r['avg_chars']:>8}c"
        f"  {r['elapsed']:>6}s"
        f"  {r['file_size_kb']:>5}KB"
    )

print(f"\n  Production file  →  corpus_B_recursive.pkl  (upload this to HF Space)")
print(f"  Ablation files   →  corpus_A_fixed.pkl, corpus_C_semantic.pkl")

# ── CELL 9: Save summary report as .txt (for your report writeup) ─────────────

summary_path = os.path.join(OUTPUT_DIR, "chunking_summary.txt")
with open(summary_path, 'w') as f:
    f.write("CHUNKING STRATEGY COMPARISON\n")
    f.write("NLP with Deep Learning — Assignment 3\n")
    f.write("="*65 + "\n\n")

    for key in ["A","B","C"]:
        r = results[key]
        f.write(f"Strategy {r['label']}: {r['description']}\n")
        f.write(f"  Total chunks   : {r['total_chunks']}\n")
        f.write(f"  Avg chunk size : {r['avg_chars']} characters\n")
        f.write(f"  Processing time: {r['elapsed']}s\n")
        f.write(f"  Output file    : corpus_{r['label']}.pkl  ({r['file_size_kb']} KB)\n")
        f.write(f"\n  Per-paper breakdown:\n")
        for fname, stats in r['per_paper'].items():
            f.write(f"    {fname:50s}  {stats['chunks']:3d} chunks  avg={stats['avg_len']}c\n")
        f.write("\n")

print(f"\n  ✓ Summary report saved to chunking_summary.txt")

# ── CELL 10: Quick sanity check on all three pkl files ───────────────────────

print(f"\n{'='*65}")
print(f"  SANITY CHECK — verifying pkl files")
print(f"{'='*65}")

for label in ["A_fixed", "B_recursive", "C_semantic"]:
    path = os.path.join(OUTPUT_DIR, f"corpus_{label}.pkl")
    with open(path, 'rb') as f:
        data = pickle.load(f)

    chunks   = data["chunks"]
    metadata = data["metadata"]

    # Pick the 10th chunk as sample
    sample_idx = min(10, len(chunks)-1)
    sample_chunk = chunks[sample_idx]
    sample_meta  = metadata[sample_idx]

    print(f"\n  corpus_{label}.pkl")
    print(f"    Chunks    : {len(chunks)}")
    print(f"    Metadata  : {len(metadata)}")
    print(f"    Sample #{sample_idx}:")
    print(f"      Title   : {sample_meta['title'][:60]}")
    print(f"      Category: {sample_meta['category']}")
    print(f"      Chunk   : {sample_meta['chunk_id']} of {sample_meta['chunk_total']}")
    print(f"      Text    : {sample_chunk[:120].strip()}...")

print(f"\n{'='*65}")
print(f"  ALL DONE. Download your pkl files from /kaggle/working/")
print(f"{'='*65}")